# Evaluation

- **J3**: Confusion matrix
- **J4**: Precision / Recall / F1 (visual)
- **J5**: Learning curve

Trains on `data.csv` if available, otherwise uses synthetic data.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import os, pandas as pd
import matplotlib.pyplot as plt
from network import Network, NetworkConfig
from optimizer import Adam
from utils import one_hot_encode
import model as model_module

## Train model

In [ ]:
np.random.seed(42)

data_path = os.path.join('..', 'data.csv')
if os.path.exists(data_path):
    df = pd.read_csv(data_path, header=None)
    y_raw = df.iloc[:, 1].values
    X_raw = df.iloc[:, 2:].values
    y_all = one_hot_encode(y_raw)
    labels = ['B (Benign)', 'M (Malignant)']
    print(f'Loaded data.csv: {len(X_raw)} samples')
else:
    X_raw = np.random.randn(200, 30)
    y_all = np.zeros((200, 2))
    y_all[np.arange(200), np.random.randint(0, 2, 200)] = 1
    labels = ['Class 0', 'Class 1']
    print('data.csv not found — using synthetic data')

idx = np.arange(len(X_raw)); np.random.shuffle(idx)
split = int(len(X_raw) * 0.8)
X_tr, X_val = X_raw[idx[:split]], X_raw[idx[split:]]
y_tr, y_val = y_all[idx[:split]], y_all[idx[split:]]

mlp = model_module.Model(
    hidden_layer_sizes=[24, 24, 24],
    learning_rate=0.001, epochs=200, batch_size=8,
    solver='adam', output_activation='softmax',
    loss='cross_entropy', weights_initializer='heUniform'
)
history = mlp.fit(X_tr, y_tr, X_val, y_val)

X_val_n = (X_val - mlp.mean_train) / mlp.std_train
val_out = mlp.predict(X_val_n)
y_pred = np.argmax(val_out, axis=1)
y_true = np.argmax(y_val, axis=1)

## J3: Confusion Matrix

In [ ]:
cm = np.zeros((2, 2), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t][p] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i][j]), ha='center', va='center', fontsize=16,
                color='white' if cm[i][j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

## J4: Precision / Recall / F1

In [ ]:
def metrics(y_true, y_pred, c):
    tp = np.sum((y_pred == c) & (y_true == c))
    fp = np.sum((y_pred == c) & (y_true != c))
    fn = np.sum((y_pred != c) & (y_true == c))
    p = tp/(tp+fp) if tp+fp else 0
    r = tp/(tp+fn) if tp+fn else 0
    f = 2*p*r/(p+r) if p+r else 0
    return p, r, f

print(f'{"Class":>15} | {"Precision":>10} | {"Recall":>10} | {"F1":>10}')
print('-' * 55)
for c in range(2):
    p, r, f = metrics(y_true, y_pred, c)
    print(f'{labels[c]:>15} | {p:>10.4f} | {r:>10.4f} | {f:>10.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(2)
precs = [metrics(y_true, y_pred, c)[0] for c in range(2)]
recs  = [metrics(y_true, y_pred, c)[1] for c in range(2)]
f1s   = [metrics(y_true, y_pred, c)[2] for c in range(2)]
w = 0.25
ax.bar(x_pos - w, precs, w, label='Precision')
ax.bar(x_pos,     recs,  w, label='Recall')
ax.bar(x_pos + w, f1s,   w, label='F1')
ax.set_xticks(x_pos); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1); ax.legend(); ax.set_title('Per-Class Metrics')
ax.axhline(0.95, color='r', ls='--', alpha=0.5, label='Recall target')
ax.axhline(0.90, color='orange', ls='--', alpha=0.5, label='Precision target')
plt.tight_layout(); plt.show()

## J5: Learning Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['loss']) + 1)

ax1.plot(epochs, history['loss'], label='Train')
ax1.plot(epochs, history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True)

ax2.plot(epochs, history['accuracy'], label='Train')
ax2.plot(epochs, history['val_accuracy'], label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True)

plt.tight_layout(); plt.show()